# Breakup CDCC in $^{8}$B+$^{208}$Pb at 83 MeV/u

We will adapt [this example from the Frescox documentation](https://www.fresco.org.uk/examples/B3-example-br-long.in).

From the documentation:

> Breakup calculations can be modeled as single-particle excitation into the continuum. In this example we show a typical CDCC calculation. It calculates the breakup of 8B into p + 7Be, under the field of 208Pb at intermediate energies. [...] The breakup of 8B has been measured many  times with the aim of extracting the proton capture rate on 7Be



In [1]:
import os
import shutil
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import inspect
import pandas as pd
import json

from tqdm import tqdm

with open("MatplotlibEsthetics.json", "r") as fptr:
    esthetics = json.load(fptr)

plt.style.use(esthetics["style"])
FONTSIZE = esthetics["fontsize"]
TICK_FONTSIZE = esthetics["tick_fontsize"]
MARKERSIZE = esthetics["markersize"]
LINEWIDTH = esthetics["linewidth"]

## BfrescoxPro

CDCC calculations are expensive. For this reason, we will use {{{Bfrescoxpro}}}, to enable parallelization with MPI. For more information, [see the docs](https://bfrescox.readthedocs.io/en/latest/advanced_users.html).

In [2]:
import bfrescoxpro
bfrescoxpro.information()

ModuleNotFoundError: No module named 'bfrescoxpro'

In [9]:
EXAMPLE_DIR = Path("./Breakup_template//")
TEMPLATE_FILE_PATH = Path(EXAMPLE_DIR / "B3-example-br.template")
INPUT_FILE_PATH = Path(EXAMPLE_DIR / "B3-example-br-long.in")

## Let's run the input file exactly as is from the website

In [10]:
with open(INPUT_FILE_PATH, "r") as temp:
    input_file = temp.read()

print("Frescox input file from https://www.fresco.org.uk/examples/B5-example-br.in:")
print("-----------------------------------")
print(input_file)

Frescox input file from https://www.fresco.org.uk/examples/B5-example-br.in:
-----------------------------------
CDCC 8B+208Pb ; nuclear and coulomb s-waves                                                    
NAMELIST
 &Fresco  hcm= 0.01 rmatch= -60.000 rintp=  0.15 rsp=   0.0
     rasym=  1000.00 accrcy= 0.0010000 
     jtmin=   0.0 jtmax=   9000.0 absend=  -50.0000 
      jump =       1      10      50     200     
      jbord=     0.0   200.0   300.0  1000.0  9000.0   
     thmin=  0.00 thmax=  20.00 thinc=  0.05  cutr=-20.00
     ips= 0.0000  it0= 0 iter=  0 iblock= 21 nnu= 24
     smallchan= 1.00E-12  smallcoup= 1.00E-12
     chans= 1 smats= 2 xstabl= 1 cdcc= 1
     elab=   656.0000    pel=1 exl=1 lab=1 lin=1 lex=1 /

 &Partition namep='8B      ' massp=  8.0000 zp=  5 nex= 21 pwf=T                
            namet='208Pb   ' masst=208.0000 zt= 82 qval=  0.1370/               
 &States jp= 1.5 ptyp=-1 ep=  0.0000  cpot=  1                                  
         jt= 0.0 ptyt= 1

## TODO use MPI

In [11]:
# Create the frescox input file by filling in the user-defined template with parameters
cfg = bfrescox.Configuration.from_template(
    INPUT_FILE_PATH,
    EXAMPLE_DIR.joinpath("frescox.in"),
    {},
    overwrite=True,
)

In [12]:
%%time
bfrescox.run_simulation(
    cfg, EXAMPLE_DIR.joinpath("frescox.out"), cwd=EXAMPLE_DIR, overwrite=True
)

CPU times: user 6.15 ms, sys: 1.02 ms, total: 7.17 ms
Wall time: 1min 50s


In [13]:
results = bfrescox.parse_fort16(EXAMPLE_DIR.joinpath("fort.16"))
results.keys()

dict_keys(['channel_1', 'channel_2', 'channel_3', 'channel_4', 'channel_5', 'channel_6', 'channel_7', 'channel_8', 'channel_9', 'channel_10', 'channel_11', 'channel_12', 'channel_13', 'channel_14', 'channel_15', 'channel_16', 'channel_17', 'channel_18', 'channel_19', 'channel_20', 'channel_21'])